# 01 Data Preparation
**Project:** Can We Detect LLM-Generated Text? A Linguistic and Stylometric Analysis of Human and Synthetic Movie Reviews

**Purpose of this notebook:** construct the human half of the paired dataset — 200 human-written IMDb reviews (100 positive, 100 negative), each 100–150 words, one review per movie, restricted to well-known movies. Each review is assigned to group A or B, and its movie title and sentiment label form the prompt used later to generate a matching LLM review.

**Output:** `human_reviews.csv`

## Step 1: Download data
Two sources are downloaded:
- IMDb Large Movie Review Dataset (Maas et al., 2011) — the human reviews. Released in 2011, therefore written before the LLM era.
- IMDb official non-commercial datasets — provide movie titles, release years and vote counts.

In [ ]:
!wget -q https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
!tar -xzf aclImdb_v1.tar.gz
!wget -q https://datasets.imdbws.com/title.basics.tsv.gz
!wget -q https://datasets.imdbws.com/title.ratings.tsv.gz
!ls

## Step 2: Load reviews
Each review file is named `{id}_{rating}.txt`. Line number `id` in `urls_pos.txt` / `urls_neg.txt` holds the IMDb link of the corresponding movie, from which the movie ID (e.g. `tt0453418`) is extracted. The ID links each review to its title and is later used as the grouping variable in cross-validation.

In [ ]:
import os, re
import pandas as pd

SEED = 42
BASE = "aclImdb/test"

def load_split(label):
    with open(f"{BASE}/urls_{label}.txt") as f:
        urls = [line.strip() for line in f]
    rows = []
    for fname in os.listdir(f"{BASE}/{label}"):
        rid, rating = fname[:-4].split("_")
        with open(f"{BASE}/{label}/{fname}", encoding="utf-8") as f:
            text = f.read()
        match = re.search(r"(tt\d+)", urls[int(rid)])
        rows.append({
            "review_id": f"{label}_{rid}",
            "label": label,
            "rating": int(rating),
            "tconst": match.group(1) if match else None,
            "text": text,
        })
    return pd.DataFrame(rows)

df = pd.concat([load_split("pos"), load_split("neg")], ignore_index=True)
print("All reviews:", len(df))
df.head()

## Step 3: Clean text and filter by length
IMDb texts contain `<br />` HTML tags, which would distort the punctuation and sentence-length features and are therefore removed. Only reviews of 100–150 words are retained — the same length range specified in the generation prompt, so that text length is controlled across conditions.

In [ ]:
def clean(text):
    text = re.sub(r"<br\s*/?>", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text"] = df["text"].apply(clean)
df["word_count"] = df["text"].str.split().str.len()
df = df[df["word_count"].between(100, 150)]
print("Reviews with 100-150 words:", len(df))

## Step 4: Add movie titles and restrict to well-known movies
A model that does not know the movie produces a generic or invented review, which would introduce an additional confound. Only movies with at least 20,000 IMDb votes are kept.

In [ ]:
basics = pd.read_csv("title.basics.tsv.gz", sep="\t", quoting=3, dtype=str,
                     na_values="\\N",
                     usecols=["tconst", "titleType", "primaryTitle", "startYear"])
ratings = pd.read_csv("title.ratings.tsv.gz", sep="\t", na_values="\\N")

df = df.merge(basics, on="tconst", how="inner")
df = df.merge(ratings[["tconst", "numVotes"]], on="tconst", how="inner")
df = df[(df["titleType"] == "movie") & (df["numVotes"] >= 20000) & df["startYear"].notna()]
print("Reviews of well-known movies:", len(df))

## Step 5: One review per movie
At most one review per movie is retained, so that no single movie dominates the dataset and each movie ID forms a group of exactly two texts (one human, one LLM) in the final dataset.

In [ ]:
df = df.sample(frac=1, random_state=SEED).drop_duplicates("tconst")
print(df["label"].value_counts())

## Step 6: Sample 100 positive + 100 negative reviews and assign groups A / B
Within each sentiment class, 50 reviews are assigned to group A and 50 to group B.
- Group A human reviews are paired with LLM A reviews
- Group B human reviews are paired with LLM B reviews

This assignment keeps the cross-LLM experiment free of data leakage: training on group A (human + LLM A) and testing on group B (human + LLM B) never reuses the same human text, and both classes remain balanced 1:1.

In [ ]:
N = 50
parts = []
for label in ["pos", "neg"]:
    pool = df[df["label"] == label]
    assert len(pool) >= 2 * N, f"Not enough {label} reviews: {len(pool)}. Lower the numVotes threshold."
    sub = pool.sample(2 * N, random_state=SEED).copy()
    sub["llm_group"] = ["A"] * N + ["B"] * N
    parts.append(sub)

human = pd.concat(parts, ignore_index=True)
print(pd.crosstab(human["label"], human["llm_group"]))
print(human["word_count"].describe())

## Step 7: Build the standardized generation prompt
One prompt template is used for both models. Only the movie title, release year and sentiment direction vary, all taken from the paired human review. Keeping the template fixed ensures that authorship, rather than instruction wording, is the variable under study.

In [ ]:
def make_prompt(row):
    sentiment = "positive" if row["label"] == "pos" else "negative"
    return (f'Write a 100-150 word {sentiment} review of the movie '
            f'"{row["primaryTitle"]}" ({row["startYear"]}), expressing your overall opinion.')

human["prompt"] = human.apply(make_prompt, axis=1)
print(human["prompt"].iloc[0])
print(human["prompt"].iloc[150])

## Step 8: Save
The resulting file contains the text, movie metadata, group assignment and generation prompt for each of the 200 human reviews.

In [ ]:
cols = ["review_id", "tconst", "primaryTitle", "startYear", "numVotes",
        "label", "rating", "llm_group", "word_count", "text", "prompt"]
human[cols].to_csv("human_reviews.csv", index=False)
print("Saved", len(human), "reviews")

from google.colab import files
files.download("human_reviews.csv")

## Step 9: Sanity check
Verifies dataset size, group balance, word-count range, HTML removal and the absence of duplicates before generation begins.

In [ ]:
print("Total:", len(human))
print(pd.crosstab(human["label"], human["llm_group"]))
print("Word count range:", human["word_count"].min(), "-", human["word_count"].max())
print("Remaining <br tags:", human["text"].str.contains("<br", case=False).sum())
print("Missing tconst:", human["tconst"].isna().sum())
print("Unique movies:", human["tconst"].nunique())
print("Duplicate review_id:", human["review_id"].duplicated().sum())
print("Prompt column exists:", "prompt" in human.columns)